# imports

In [ ]:
import sionna.rt as rt

import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np
import tensorflow as tf
from sim import build_rx_path, setup_drone_meshes, drone_mesh_z_offset
from sensing import (
    plot_range_doppler, ground_truth_range_velocity, plot_range_angle,
    world_to_spherical, detect_targets, score_sensing, bistatic_rx_range,
)
from jcas_drop import run_jcas_drop, plot_jcas_kpis

from sionna.rt import load_scene, PlanarArray, Camera

print(f"Mitsuba variant: {mi.variant()}")
print(tf.config.list_physical_devices('GPU'))

# Params

In [ ]:
preview = False
tx_power_dbm = 30       # total tx power in dBm, split across users
csi_error_std = 0.00    # stddev of simulated per-entry CSI estimation error (0 = perfect CSI)

# --- Sensing ---
sensing_on = True
plots_on = True         # graphing only; the text metrics always print
max_range_m = 250       # cube range trim; each receiver keeps one cube per step

# create scene

In [ ]:
import ofdm_config

# "3.5", "10" or "28" -- rebinds carrier, bandwidth, numerology and array together
ofdm_config.select_band("28")

scene = load_scene(rt.scene.munich, merge_shapes=False)
scene.frequency = ofdm_config.CARRIER_FREQUENCY

scene.tx_array = PlanarArray(num_rows=ofdm_config.ARRAY_ROWS, num_cols=ofdm_config.ARRAY_COLS,
                             pattern="tr38901", polarization="V")
scene.rx_array = scene.tx_array


In [ ]:
cameras = {
    "cam2": Camera(position=[-56, -42, 220], look_at=[-56, -42, 0])
}

# One UAV up the same Munich street canyon as 1_drone.ipynb, flying the last 3 s of
# its path: starting next to the BS, it would sit on every tx-rx baseline.
uas1 = rt.Receiver(name="uas_1", position=[-58.2, -46.4, 15.0], orientation=[0.0, 0.0, 0.0])
uas1.color = [1, 1, 0]
scene.add(uas1)

num_drones = 1
num_steps = 3           # each step is one solve per receiver, so ~3x a monostatic step
dt = 1

start_points = tf.constant([
    [-58.2, -46.4, 15.0],  # mid-street, ~21 m from the BS
], dtype=tf.float32)

base_velocities = tf.constant([
    [[2.4, 8.8, 0], [2.4, 8.8, 0], [2.4, 8.8, 0]],
], dtype=tf.float32)  # [N, num_steps, 3] -- 9.1 m/s from (-58,-46) to (-51,-20)

rx_path = build_rx_path(start_points, base_velocities, num_steps, dt, on_mismatch="error")

# The BS transmits only; three receive-only sensing nodes listen at 12 m, below the
# drone, so it stays inside the zenith 0-90 deg scan. Each has LOS to the whole flight,
# and the drone's range sum stays >= 10 m past each baseline, clear of the 6 m guard.
tx_position = [-69.0, -64.0, 12.0]
look_at_point = [-55.0, -40.0, 15.0]
rx_positions = [
    [-56.0, -61.0, 12.0],  # across from the BS: 13 m baseline, bistatic angle <= 40 deg
    [-53.0, -53.0, 12.0],  # beside the flight's start: 19 m, <= 68 deg
    [-59.0, -84.0, 12.0],  # down the street behind the BS: 22 m, <= 30 deg
]
rx_look_at = [-56.0, -38.0, 15.0]  # receivers face the middle of the flight


In [ ]:
# metal cube per drone: comms multipath scatterer and radar target alike
drone_radius_m = 0.25
drone_meshes = setup_drone_meshes(scene, drone_radius_m=drone_radius_m)

In [ ]:
# One run_jcas_drop() per step: the first position is the transmitter, the rest are
# the sensing receivers, solved one after another. Comms is as in 1_drone.ipynb.
drone_names = list(scene.receivers)
tx_power_w = 10 ** (tx_power_dbm / 10) / 1000  # dBm → W

sensing_cubes = []            # [T][R] RadarCube per timestep, per receiver
sensing_ground_truth = []     # [T][R] (half range sum, half its rate) per drone
sensing_ground_truth_pos = [] # [T] drone world positions
jcas_results = []             # [T] JCASDropResult per timestep

if sensing_on:
    for t in range(num_steps):
        truths = [[] for _ in rx_positions]
        positions = []
        for i, name in enumerate(drone_names):
            pos = rx_path[i, t, :].numpy()
            scene.receivers[name].position = pos.tolist()
            mesh_pos = pos.copy()
            mesh_pos[2] += drone_mesh_z_offset(drone_radius_m)
            drone_meshes[name].position = mesh_pos.tolist()
            vel = (rx_path[i, t + 1, :] - rx_path[i, t, :]).numpy() / dt
            drone_meshes[name].velocity = vel.tolist()
            for r, rx in enumerate(rx_positions):
                truths[r].append(ground_truth_range_velocity(tx_position, pos, vel, rx_position=rx))
            positions.append(pos)
        sensing_ground_truth.append(truths)
        sensing_ground_truth_pos.append(positions)

        result = run_jcas_drop(
            scene, bs_position=[tx_position] + rx_positions, look_at=look_at_point,
            rx_names=drone_names, tx_power_w=tx_power_w, csi_error_std=csi_error_std,
            rx_look_at=rx_look_at, max_range_m=max_range_m,
        )
        jcas_results.append(result)
        sensing_cubes.append(result.radar_cubes)

        comms = result.comms[drone_names[0]]
        print(f"Step t={t}: comms SNR={comms.snr_db:.1f} dB, BER={comms.ber:.2e}, "
              f"throughput={comms.throughput_mbps:.1f} Mbps")
        for r, cube in enumerate(result.radar_cubes):
            peak = cube.range_bins[cube.data.max(axis=(0, 1)).argmax()]
            print(f"  rx_{r + 1}: peak power {cube.data.max():.3e}, peak half range sum {peak:.1f} m "
                  f"(truth {truths[r][0][0]:.1f} m)")


In [ ]:
# per-receiver detection accuracy, false alarms and position error
if sensing_on and sensing_cubes:
    for r, rx in enumerate(rx_positions):
        print(f"rx_{r + 1} at {rx}")
        print(score_sensing([cubes[r] for cubes in sensing_cubes], sensing_ground_truth_pos,
                            tx_position, rx_position=rx))

In [ ]:
# SNR, BER and throughput, measured off the decoded OFDM frames
if plots_on and jcas_results:
    plot_jcas_kpis(jcas_results, scene)

In [ ]:
# rows are receivers, columns are steps; range is half the bistatic range sum
if plots_on and sensing_on and sensing_cubes:
    nrows, ncols = len(rx_positions), len(sensing_cubes)
    fig = plt.figure(figsize=(6 * ncols, 5 * nrows))
    for r in range(nrows):
        for t_idx in range(ncols):
            ax = fig.add_subplot(nrows, ncols, r * ncols + t_idx + 1)
            plot_range_doppler(
                sensing_cubes[t_idx][r],
                max_range_m=max_range_m,
                ax=ax,
                title=f"Range-Doppler — rx_{r + 1}, step {t_idx}",
                ground_truth=sensing_ground_truth[t_idx][r],
            )
    plt.tight_layout()
    plt.show()


In [ ]:
sensing_detections = []  # [T][R] RadarPointCloud, reused by the map below

if plots_on and sensing_on and sensing_cubes:
    nrows, ncols = len(rx_positions), len(sensing_cubes)
    fig = plt.figure(figsize=(6 * ncols, 5 * nrows))
    for t_idx in range(ncols):
        sensing_detections.append([detect_targets(cube) for cube in sensing_cubes[t_idx]])
    for r, rx in enumerate(rx_positions):
        for t_idx in range(ncols):
            ax = fig.add_subplot(nrows, ncols, r * ncols + t_idx + 1)
            gt_range_azimuth = [world_to_spherical(tx_position, pos, rx_position=rx)[:2]
                                for pos in sensing_ground_truth_pos[t_idx]]
            plot_range_angle(
                sensing_cubes[t_idx][r],
                ground_truth=gt_range_azimuth,
                detections=sensing_detections[t_idx][r],
                max_range_m=max_range_m,
                ax=ax,
                title=f"Range-Azimuth — rx_{r + 1}, step {t_idx}",
            )
    plt.tight_layout()
    plt.show()

In [ ]:
# Top-down map: every receiver's detections placed on its range-sum ellipsoid
if plots_on and sensing_on and sensing_detections:
    fig, ax = plt.subplots(figsize=(8, 8))
    colors = ["tab:blue", "tab:orange", "tab:purple"]
    for r, rx in enumerate(rx_positions):
        xy = []
        for dets in sensing_detections:
            for p in dets[r].points:
                half_range_sum = float(np.linalg.norm(p.position))
                rx_range = bistatic_rx_range(half_range_sum, p.position, tx_position, rx)
                xy.append(np.asarray(rx) + rx_range * p.position / half_range_sum)
        xy = np.array(xy).reshape(-1, 3)
        ax.scatter(xy[:, 0], xy[:, 1], s=15, color=colors[r], alpha=0.6, label=f"rx_{r + 1} detections")
        ax.scatter(*rx[:2], marker="^", s=120, color=colors[r], edgecolor="k")
    ax.scatter(*tx_position[:2], marker="s", s=120, color="red", edgecolor="k", label="tx")
    truth = np.array([p for step in sensing_ground_truth_pos for p in step])
    ax.plot(truth[:, 0], truth[:, 1], "g+-", ms=14, label="drone truth")
    ax.set_xlim(-95, -15)
    ax.set_ylim(-95, -15)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.legend(loc="upper left")
    plt.show()

In [ ]:
scene.preview()